# Sinhala Document Understanding with Donut — Kaggle GPU notebook

**Runs on Kaggle with a GPU only.** See `Guide/rules.md` for the compatibility checklist.

## What changed in this version (Session 6)

Four *fatal* bugs were found and fixed. Three of them were introduced by earlier
"defensive" edits; all four independently forced validation metrics to exactly 0.0
while training loss fell normally.

| # | Bug | Effect | Fix |
|---|-----|--------|-----|
| 1 | `decoder_start_token_id` set only on `model.config` | `generate()` reads **only** `generation_config`, so decoding began at `<s>` (id 0) — a token never seen at position 0 in training. Loss looked healthy; generation was garbage. | Set on **both**, assert it, re-assert in a callback |
| 2 | `no_repeat_ngram_size=3` | Hard `-inf` ban on repeating any 3-gram. Our grammar needs `</s_answer><sep/><s_question>` ~11-17x per document → a correct answer was **mathematically unreachable** | Removed (`0`) |
| 3 | `repetition_penalty=1.3` | Penalised exactly the structural tags that must repeat | Removed (`1.0`) |
| 4 | eval generation capped at 640 tokens | Longer documents could never reach their closing tags → unparseable → 0 | Use the full measured length |

Plus two substantive quality fixes, both **measured, not assumed**:

* **Tokenizer.** donut-base's own vocabulary destroys **86.6%** of Sinhala characters
  (34.7% `<unk>`; `'පදිංචි ලිපිනය'` → `'පද<unk> <unk>'`), so `donut_native` was never viable.
  The old workaround — swapping in SinBERT and *shrinking* the embedding matrix — **truncates**
  donut's pretrained rows and scrambles ~106M parameters. This version instead **grows** donut's
  own vocabulary with the Sinhala pieces XLM-R already has. Result: `<unk>` **34.7% → 0.0%**,
  round-trip **24.8% → 100.0%**, and *shorter* targets (mean 516 → 354 tokens).
* **Data.** Targets were ~32% pairwise-inverted vs. reading order. `regenerate_donut_dataset.py`
  rebuilds them in reading order from the raw SinFUND boxes (**31.8% → 2.5%** inversions) and
  recovers pairs the old conversion dropped.

## Before running

* Accelerator **GPU**, Internet **ON**, Kaggle secret **`HF_TOKEN`**.
* Attach the dataset. Prefer **`SinFundDonutV2`** (produced by `regenerate_donut_dataset.py`).
  `SinFundDonut` still works but the notebook will warn that field order is scrambled.


In [ ]:
# ============================================================
# CELL 1 — ENVIRONMENT, CONFIG, GPU, HF AUTH, DATASET DISCOVERY
# ============================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # single GPU; avoids DataParallel issues

import gc, glob, json, random, re, unicodedata
import numpy as np
import torch

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------
# "donut_extended_vocab"  -> RECOMMENDED. Donut encoder+decoder, donut's OWN
#                            vocabulary GROWN with the Sinhala pieces XLM-R has.
#                            Pretrained embeddings stay aligned; only new rows are new.
# "donut_sinhala_vocab"   -> DEPRECATED. Swaps in SinBERT's vocab and SHRINKS the
#                            embedding matrix, which truncates/scrambles ~106M
#                            pretrained params. Kept only for ablation.
# "donut_native"          -> BROKEN for Sinhala (86.6% of characters become <unk>).
#                            Kept as a documented negative control for the thesis.
MODEL_VARIANT = "donut_extended_vocab"

DONUT_BASE_ID = "naver-clova-ix/donut-base"
XLMR_ID       = "xlm-roberta-base"                # donor for Sinhala vocabulary pieces
TOKENIZER_ID  = "NLPC-UOM/SinBERT-large"          # only for donut_sinhala_vocab
TROCR_HUB_ID  = "danush99/Model_TrOCR-Sin-Handwritten-Text"

TASK_START_TOKEN, TASK_END_TOKEN = "<s_gt_parse>", "</s_gt_parse>"

OUTPUT_DIR       = "./donut_sinhala_output"
FINAL_MODEL_PATH = "./donut_sinhala_final"

# (height, width). Official Donut CORD fine-tuning uses [1280, 960]; our pages are
# ~2200x3000 with a median text line of only ~45px, so at the old [960, 640] a line
# was ~15px tall — too small for handwritten Sinhala.
IMAGE_SIZE = [1280, 960]

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ------------------------------------------------------------
# GPU (fail fast)
# ------------------------------------------------------------
print("=" * 70); print("ENVIRONMENT"); print("=" * 70)
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda,
      "| available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA unavailable. This notebook runs only on Kaggle with a GPU accelerator "
        "(Settings -> Accelerator -> GPU). See Guide/rules.md.")
print("GPU:", torch.cuda.get_device_name(0),
      "| memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

# ------------------------------------------------------------
# HUGGING FACE AUTH (Kaggle Secrets)
# ------------------------------------------------------------
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("\nHF authentication: configured")
except Exception as e:
    print("\nWarning: HF_TOKEN unavailable (", e, ") -- public models only.")

# ------------------------------------------------------------
# DATASET DISCOVERY (never hardcode the mount path)
# ------------------------------------------------------------
def find_dataset_root():
    """Prefer SinFundDonutV2 (reading-order corrected); fall back to SinFundDonut."""
    for marker in ("SinFundDonutV2", "SinFundDonut"):
        for root in ("/kaggle/input", "."):
            if not os.path.isdir(root):
                continue
            for path in glob.glob(os.path.join(root, "**", marker), recursive=True):
                if os.path.exists(os.path.join(path, "train", "metadata.jsonl")):
                    return path, marker
    return None, None

DATASET_ROOT, DATASET_KIND = find_dataset_root()
if DATASET_ROOT is None:
    print("\nContents of /kaggle/input:")
    for p in glob.glob("/kaggle/input/*"):
        print(" -", p)
    raise FileNotFoundError(
        "Could not find SinFundDonutV2 or SinFundDonut under /kaggle/input. "
        "Attach it via Notebook Settings -> Add Input -> Datasets.")

TRAIN_PATH      = os.path.join(DATASET_ROOT, "train")
VALIDATION_PATH = os.path.join(DATASET_ROOT, "validation")
print("\nDataset:", DATASET_KIND, "at", DATASET_ROOT)
if DATASET_KIND == "SinFundDonut":
    print("  WARNING: this is the ORIGINAL dataset. Its question/answer order is ~32%")
    print("  pairwise-inverted vs. document reading order, which makes the target")
    print("  sequence much harder to learn. Run regenerate_donut_dataset.py locally and")
    print("  upload SinFundDonutV2 for a materially better result.")
print("=" * 70)


In [ ]:
# ============================================================
# CELL 2 — LOAD RECORDS + DONUT SCHEMA
# ============================================================
# Ground truth uses a SMALL FIXED tag set (form / question / answer) with all the
# variable Sinhala text as tag CONTENT. Putting the variable field labels in tag
# NAMES (the original dataset's shape) would mint a near-unique special token per
# document, which no model can learn to emit.

print("=" * 70); print("DATA"); print("=" * 70)

def load_split(split_path):
    jsonl = os.path.join(split_path, "metadata.jsonl")
    if not os.path.exists(jsonl):
        raise FileNotFoundError(jsonl)
    out = []
    with open(jsonl, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            gt  = row["ground_truth"]
            gt  = json.loads(gt) if isinstance(gt, str) else gt
            parse = gt.get("gt_parse", gt)
            if "form" in parse:                       # SinFundDonutV2 (already correct)
                entries = [{"question": str(e.get("question", "")).strip(),
                            "answer":   str(e.get("answer", "")).strip()}
                           for e in parse["form"]]
            else:                                     # legacy flat {label: value}
                entries = []
                for q, a in parse.items():
                    for v in (a if isinstance(a, list) else [a]):
                        entries.append({"question": str(q).strip(),
                                        "answer":   str(v).strip()})
            out.append({"image_path": os.path.join(split_path, row["file_name"]),
                        "entries": entries})
    return out

train_records = load_split(TRAIN_PATH)
val_records   = load_split(VALIDATION_PATH)

def json2token(obj, sort_json_key=True):
    """Donut's official ground-truth -> tag-sequence conversion, so that
    DonutProcessor.token2json can invert it at evaluation time."""
    if isinstance(obj, dict):
        keys = sorted(obj.keys(), reverse=True) if sort_json_key else list(obj.keys())
        return "".join(f"<s_{k}>" + json2token(obj[k], sort_json_key) + f"</s_{k}>"
                       for k in keys)
    if isinstance(obj, list):
        return "<sep/>".join(json2token(i, sort_json_key) for i in obj)
    return str(obj)

STRUCTURAL_TOKENS = ["<s_gt_parse>", "</s_gt_parse>", "<s_form>", "</s_form>",
                     "<s_question>", "</s_question>", "<s_answer>", "</s_answer>",
                     "<sep/>"]

n_pairs = sum(len(r["entries"]) for r in train_records + val_records)
print(f"train docs={len(train_records)}  val docs={len(val_records)}  q/a pairs={n_pairs}")
print("structural tokens:", STRUCTURAL_TOKENS)
print("\nexample target (first 400 chars):")
print(json2token({"gt_parse": {"form": train_records[0]["entries"]}})[:400])
print("=" * 70)


In [ ]:
# ============================================================
# CELL 3 — TOKENIZER: GROW DONUT'S VOCABULARY FOR SINHALA
# ============================================================
# Every number in this cell was measured on this dataset, not assumed.
#
# donut-base's tokenizer is a SentencePiece **Unigram** model (vocab 57,525,
# byte_fallback = False) -- a PRUNED XLM-RoBERTa vocab that kept only 280 pieces
# containing Sinhala. Measured on our targets it produces 34.7% <unk> and loses
# 86.6% of all Sinhala characters ('පදිංචි ලිපිනය' -> 'පද<unk> <unk>').
#
# The previous workaround swapped in SinBERT's 52k vocab and called
# resize_token_embeddings(52009). Shrinking TRUNCATES: donut's pretrained rows
# 0..52008 are kept and silently reinterpreted under unrelated ids -- ~106M
# parameters of misaligned weights to relearn from 80 documents.
#
# Instead we GROW donut's own vocabulary. donut's vocab is a strict subset of
# XLM-R's, with the same Unigram model, the same Precompiled normalizer, the same
# Metaspace pre-tokenizer and an identical score range, so XLM-R's Sinhala pieces
# are drop-in. Because the vocabulary only grows, resize_token_embeddings APPENDS
# rows and every pretrained row keeps its meaning.

from transformers import AutoTokenizer, DonutProcessor, PreTrainedTokenizerFast
from tokenizers import Tokenizer

print("=" * 70); print(f"TOKENIZER  (MODEL_VARIANT={MODEL_VARIANT!r})"); print("=" * 70)

# donut's NMT_NFKC normalizer rewrites ZWJ (U+200D) to a SPACE, which breaks every
# Sinhala conjunct (ශ්\u200dරී -> ශ් රී). ZWJ occurs in 98/100 of our documents. These
# private-use sentinels pass through the normalizer untouched, so we substitute on
# the way in and restore on the way out.
ZWJ, ZWNJ = "\u200d", "\u200c"           # zero-width joiner / non-joiner
SENT_ZWJ, SENT_ZWNJ = "\ue000", "\ue001"  # private-use; survive the normalizer
enc_txt = lambda s: s.replace(ZWJ, SENT_ZWJ).replace(ZWNJ, SENT_ZWNJ)
dec_txt = lambda s: s.replace(SENT_ZWJ, ZWJ).replace(SENT_ZWNJ, ZWNJ)

processor = DonutProcessor.from_pretrained(DONUT_BASE_ID)
donut_tok = AutoTokenizer.from_pretrained(DONUT_BASE_ID)
_normalizer = donut_tok.backend_tokenizer.normalizer

def canon(s):
    """Canonicalise ground truth through the tokenizer's OWN normalizer so the
    target is exactly what the pipeline can reproduce (round-trip -> 100%)."""
    return re.sub(r"\s+", " ", dec_txt(_normalizer.normalize_str(enc_txt(s)))).strip()

for rec in train_records + val_records:
    for e in rec["entries"]:
        e["question"] = canon(e["question"])
        e["answer"]   = canon(e["answer"])
    rec["schema"] = {"gt_parse": {"form": rec["entries"]}}

ALL_FIELDS = [v for r in train_records + val_records for e in r["entries"]
              for v in (e["question"], e["answer"])]

def measure(tok, fields, unk_id=3, label=""):
    exact = toks = unk = 0
    sin = lambda s: sum(1 for c in s if "඀" <= c <= "෿")
    s_in = s_out = 0
    for f in fields:
        ids = tok.encode(enc_txt(f), add_special_tokens=False).ids
        toks += len(ids); unk += sum(1 for i in ids if i == unk_id)
        out = dec_txt(tok.decode(ids))
        exact += (out == f); s_in += sin(f); s_out += sin(out)
    print(f"  {label:24s} exact={100*exact/max(len(fields),1):6.2f}%  "
          f"unk={100*unk/max(toks,1):6.3f}%  sinhala_kept={100*s_out/max(s_in,1):6.2f}%  tokens={toks}")
    return exact / max(len(fields), 1)

def build_extended_tokenizer(donut_tokenizer, xlmr_tokenizer, corpus, extra_tokens):
    dj = json.loads(donut_tokenizer.backend_tokenizer.to_str())
    xj = json.loads(xlmr_tokenizer.backend_tokenizer.to_str())
    vocab = dj["model"]["vocab"]; have = {p for p, _ in vocab}; n0 = len(vocab)
    is_sin = lambda s: any("඀" <= c <= "෿" for c in s)

    n_sin = 0
    for piece, score in xj["model"]["vocab"]:          # 1. Sinhala pieces donut lost
        if is_sin(piece) and piece not in have:
            vocab.append([piece, score]); have.add(piece); n_sin += 1

    chars = set()                                       # 2. character-level fallback
    for t in corpus:
        chars.update(enc_txt(t))
    n_ch = 0
    for c in sorted(chars):
        if c not in have and not c.isspace():
            vocab.append([c, -20.5]); have.add(c); n_ch += 1

    n_sent = 0                                          # 3. ZWJ/ZWNJ sentinels
    for s in (SENT_ZWJ, SENT_ZWNJ):
        if s not in have:
            vocab.append([s, -15.0]); have.add(s); n_sent += 1

    n_st = 0                                            # 4. structural tags
    for t in extra_tokens:
        if t not in have:
            vocab.append([t, -1.0]); have.add(t); n_st += 1

    # Drop donut's legacy added_tokens (<sep/>, <s_iitcdip>, <s_synthdog>): their
    # fixed ids collide with the grown vocab, AND added tokens are matched outside
    # the Unigram model so the Metaspace decoder injects a spurious space at every
    # boundary ('<sep/></s_form>' -> '<sep/> </s_form>'). We emit <sep/> ~17x per
    # target, so that would corrupt every sequence. As ordinary pieces they are clean.
    dj["added_tokens"] = [a for a in dj.get("added_tokens", [])
                          if a.get("content") in {"<s>", "<pad>", "</s>", "<unk>"}]
    print(f"  vocab {n0} -> {len(vocab)}  (+{n_sin} sinhala, +{n_ch} chars, "
          f"+{n_sent} sentinels, +{n_st} tags)")
    return Tokenizer.from_str(json.dumps(dj))

if MODEL_VARIANT == "donut_extended_vocab":
    xlmr = AutoTokenizer.from_pretrained(XLMR_ID)
    print("\nBaseline vs extended (measured on this dataset's fields):")
    measure(donut_tok.backend_tokenizer, ALL_FIELDS, label="donut-base (unmodified)")
    ext = build_extended_tokenizer(donut_tok, xlmr, ALL_FIELDS, STRUCTURAL_TOKENS)
    acc = measure(ext, ALL_FIELDS, label="donut + Sinhala")
    tokenizer = PreTrainedTokenizerFast(tokenizer_object=ext, unk_token="<unk>",
                                        pad_token="<pad>", bos_token="<s>", eos_token="</s>")
    if acc < 0.99:
        raise RuntimeError(f"Extended tokenizer round-trip only {acc:.2%} -- refusing to "
                           "train on lossy targets. Investigate before continuing.")

elif MODEL_VARIANT == "donut_sinhala_vocab":
    print("\nWARNING: this variant SHRINKS donut's embedding matrix (57,525 -> ~52,009),")
    print("truncating and scrambling ~106M pretrained parameters. Ablation only.")
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, use_fast=False, token=HF_TOKEN)
    if tokenizer.eos_token is None:
        tokenizer.add_special_tokens({"eos_token": "</s>"})
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": "<pad>"})
    tokenizer.add_tokens(STRUCTURAL_TOKENS)

elif MODEL_VARIANT == "donut_native":
    print("\nWARNING: donut-base's vocabulary cannot represent Sinhala "
          "(measured: 34.7% <unk>, 86.6% of Sinhala characters lost).")
    print("This variant exists as a documented negative control only.")
    tokenizer = donut_tok
    tokenizer.add_tokens(STRUCTURAL_TOKENS)
    measure(tokenizer.backend_tokenizer, ALL_FIELDS, label="donut_native (broken)")

else:
    raise ValueError(f"Unknown MODEL_VARIANT: {MODEL_VARIANT!r}")

processor.tokenizer = tokenizer

# Every structural tag must be exactly ONE id IN CONTEXT, or the grammar cannot be
# generated. We test in context deliberately: the Metaspace pre-tokenizer prepends a
# word-start marker "\u2581" to the beginning of ANY string, so tokenizing a tag in
# isolation always yields ["\u2581", tag] -- that leading marker is an artifact of the
# isolated test, not a defect. (It also means each target sequence begins
# ["\u2581", "<s_gt_parse>", ...]; harmless, and clean_generated() strips it.)
for t in STRUCTURAL_TOKENS:
    toks = tokenizer.convert_ids_to_tokens(
        tokenizer("\u0d85" + t + "\u0d85", add_special_tokens=False)["input_ids"])
    assert t in toks, f"structural token {t!r} is not a single id in context: {toks}"
    assert tokenizer.convert_tokens_to_ids(t) != tokenizer.unk_token_id, \
        f"structural token {t!r} maps to <unk>"

TASK_START_ID = tokenizer.convert_tokens_to_ids(TASK_START_TOKEN)
print(f"\nvocab={len(tokenizer)}  {TASK_START_TOKEN}={TASK_START_ID}  "
      f"pad={tokenizer.pad_token_id} eos={tokenizer.eos_token_id} unk={tokenizer.unk_token_id}")
print("=" * 70)


In [ ]:
# ============================================================
# CELL 4 — MODEL
# ============================================================
from transformers import (VisionEncoderDecoderConfig, VisionEncoderDecoderModel,
                          AutoTokenizer)

print("=" * 70); print("MODEL"); print("=" * 70)

# image_size MUST be set on the config BEFORE from_pretrained. Setting
# model.config.encoder.image_size afterwards is a no-op -- the encoder module has
# already been built -- and it also writes a false image_size into every checkpoint.
config = VisionEncoderDecoderConfig.from_pretrained(DONUT_BASE_ID)
config.encoder.image_size = IMAGE_SIZE

if MODEL_VARIANT == "hybrid":
    donut_model = VisionEncoderDecoderModel.from_pretrained(DONUT_BASE_ID, config=config)
    trocr = VisionEncoderDecoderModel.from_pretrained(TROCR_HUB_ID, token=HF_TOKEN)
    hybrid_cfg = VisionEncoderDecoderConfig.from_encoder_decoder_configs(
        donut_model.encoder.config, trocr.decoder.config)
    model = VisionEncoderDecoderModel(config=hybrid_cfg,
                                      encoder=donut_model.encoder, decoder=trocr.decoder)
    eh, dh = donut_model.encoder.config.hidden_size, trocr.decoder.config.hidden_size
    if eh != dh:
        model.enc_to_dec_proj = torch.nn.Linear(eh, dh)
    del donut_model, trocr; gc.collect(); torch.cuda.empty_cache()
else:
    model = VisionEncoderDecoderModel.from_pretrained(DONUT_BASE_ID, config=config)

# donut-base stores embed_tokens.weight and lm_head.weight as SEPARATE tensors even
# though its config claims they are tied. Leaving the claim uncorrected makes
# save_model() skip writing lm_head.weight, and reloading then silently drops the
# trained output projection ("missing keys: ['decoder.lm_head.weight']").
model.config.tie_word_embeddings = False

old_vocab = model.decoder.get_input_embeddings().weight.shape[0]
model.decoder.resize_token_embeddings(len(tokenizer))
new_vocab = model.decoder.get_input_embeddings().weight.shape[0]
model.config.vocab_size = model.config.decoder.vocab_size = len(tokenizer)
print(f"decoder embeddings: {old_vocab} -> {new_vocab}", end="  ")
print("(GROWN - pretrained rows preserved)" if new_vocab >= old_vocab
      else "(!! SHRUNK - pretrained rows truncated and scrambled !!)")

processor.image_processor.size = {"height": IMAGE_SIZE[0], "width": IMAGE_SIZE[1]}

# ------------------------------------------------------------
# TOKEN IDS -- must be set in BOTH places
# ------------------------------------------------------------
# TRAINING reads model.config: VisionEncoderDecoderModel.forward builds decoder
#   inputs via shift_tokens_right(labels, config.pad_token_id, config.decoder_start_token_id)
# GENERATION reads model.generation_config ONLY. The chain is
#   generate(kwarg) > generation_config.decoder_start_token_id > generation_config.bos_token_id
# and model.config is NOT in it. Setting only model.config is what made generation
# start at <s> (id 0) while training started at <s_gt_parse> -- healthy loss,
# zero metrics. See Guide/learnings.md Session 6.
for cfg in (model.config, model.generation_config):
    cfg.decoder_start_token_id = TASK_START_ID
    cfg.pad_token_id = tokenizer.pad_token_id
    cfg.eos_token_id = tokenizer.eos_token_id
model.generation_config.bos_token_id = TASK_START_ID     # neutralise the bos fallback

# Structured decoding: no repetition controls (they make the grammar unreachable),
# greedy, ban <unk> as official Donut does.
gc_ = model.generation_config
gc_.no_repeat_ngram_size = 0
gc_.repetition_penalty   = 1.0
gc_.num_beams            = 1
gc_.do_sample            = False
gc_.early_stopping       = False
gc_.forced_eos_token_id  = None
gc_.bad_words_ids        = [[tokenizer.unk_token_id]]

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False        # required for gradient checkpointing
model = model.cuda()

assert model.generation_config.decoder_start_token_id == TASK_START_ID
print(f"decoder_start_token_id = {TASK_START_ID} ({TASK_START_TOKEN}) on config AND generation_config")
print("image_size:", model.config.encoder.image_size, "| device:", next(model.parameters()).device)
print("=" * 70)


In [ ]:
# ============================================================
# CELL 5 — DATASET, MAX_LENGTH, SMOKE TEST
# ============================================================
from PIL import Image
from torch.utils.data import Dataset as TorchDataset

print("=" * 70); print("DATASET"); print("=" * 70)

def target_sequence(rec):
    # The task token is BOTH decoder_start_token_id and the first label token,
    # matching the reference HF Donut fine-tuning notebooks. The metric code
    # strips everything up to and including it.
    return json2token(rec["schema"]) + tokenizer.eos_token

lens = [len(tokenizer(enc_txt(target_sequence(r)), add_special_tokens=False)["input_ids"])
        for r in train_records + val_records]
MAX_LENGTH = int(min(1280, max(256, np.ceil((max(lens) + 8) / 32.0) * 32)))
print(f"target tokens: min={min(lens)} mean={np.mean(lens):.0f} max={max(lens)} "
      f"p95={int(np.percentile(lens,95))}  ->  MAX_LENGTH={MAX_LENGTH}")
if max(lens) + 8 > MAX_LENGTH:
    print("WARNING: some targets will be TRUNCATED -- raise the 1280 cap.")

class DonutDataset(TorchDataset):
    def __init__(self, records, processor, max_length):
        self.records, self.processor, self.max_length = records, processor, max_length
    def __len__(self):
        return len(self.records)
    def __getitem__(self, idx):
        rec = self.records[idx]
        image = Image.open(rec["image_path"]).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze(0)
        labels = self.processor.tokenizer(
            enc_txt(target_sequence(rec)), add_special_tokens=False,
            max_length=self.max_length, padding="max_length", truncation=True,
            return_tensors="pt")["input_ids"].squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels}

train_dataset      = DonutDataset(train_records, processor, MAX_LENGTH)
validation_dataset = DonutDataset(val_records,   processor, MAX_LENGTH)
print(f"train={len(train_dataset)}  val={len(validation_dataset)}")

s = train_dataset[0]
print("\npixel_values:", tuple(s["pixel_values"].shape), "| labels:", tuple(s["labels"].shape))
disp = s["labels"].clone(); disp[disp == -100] = tokenizer.pad_token_id
print("decoded target:", dec_txt(tokenizer.decode(disp, skip_special_tokens=False))[:300])

print("\nforward/backward smoke test...")
model.train()
out = model(pixel_values=s["pixel_values"].unsqueeze(0).cuda(),
            labels=s["labels"].unsqueeze(0).cuda())
print("  loss:", out.loss.item())
out.loss.backward()
model.zero_grad(set_to_none=True)
del out; torch.cuda.empty_cache()
print("  PASSED")
print("=" * 70)


In [ ]:
# ============================================================
# CELL 6 — METRICS
# ============================================================
# Field-level P/R/F1 and exact-match all floor at exactly 0.0 whenever generation
# is unparseable, so they cannot distinguish "pipeline broken" from "learning
# slowly" -- which is what hid the real bug for three sessions. Official Donut
# reports normalized edit distance on the RAW decoded string, which gives a
# continuous signal from epoch 1. We report both.
from collections import Counter

print("=" * 70); print("METRICS"); print("=" * 70)

def clean_generated(text):
    for t in (tokenizer.eos_token, tokenizer.pad_token, tokenizer.bos_token):
        if t:
            text = text.replace(t, "")
    text = re.sub(r"^.*?" + re.escape(TASK_START_TOKEN), "", text, count=1)
    return text.replace(TASK_END_TOKEN, "").strip()

def parse_pairs(text):
    try:
        parsed = processor.token2json(clean_generated(text))
    except Exception:
        parsed = {}
    entries = parsed.get("form", []) if isinstance(parsed, dict) else []
    if isinstance(entries, dict):
        entries = [entries]
    return [(str(e.get("question", "")).strip(), str(e.get("answer", "")).strip())
            for e in entries if isinstance(e, dict)]

def levenshtein(a, b):
    if a == b:
        return 0
    if not a or not b:
        return len(a) or len(b)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb))
        prev = cur
    return prev[len(b)]

def norm_edit_sim(a, b):
    return 1.0 - levenshtein(a, b) / max(len(a), len(b), 1)

def score_predictions(preds, refs):
    matched = p_tot = r_tot = exact = 0
    raw_sims, pair_sims = [], []
    for p_txt, r_txt in zip(preds, refs):
        pp, rp = parse_pairs(p_txt), parse_pairs(r_txt)
        pc, rc = Counter(pp), Counter(rp)
        matched += sum((pc & rc).values()); p_tot += len(pp); r_tot += len(rp)
        exact += (pc == rc)
        # continuous, parser-independent signal on the raw strings
        raw_sims.append(norm_edit_sim(clean_generated(p_txt), clean_generated(r_txt)))
        pair_sims.append(norm_edit_sim("\n".join(f"{q}: {a}" for q, a in sorted(pp)),
                                       "\n".join(f"{q}: {a}" for q, a in sorted(rp))))
    prec = matched / p_tot if p_tot else 0.0
    rec  = matched / r_tot if r_tot else 0.0
    return {"precision": prec, "recall": rec,
            "f1": 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0,
            "exact_match": exact / max(len(preds), 1),
            "edit_similarity": float(np.mean(pair_sims)) if pair_sims else 0.0,
            "raw_similarity": float(np.mean(raw_sims)) if raw_sims else 0.0}

def compute_metrics(eval_preds):
    pred_ids, label_ids = eval_preds
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]
    # Trainer pads BOTH predictions and labels with -100 when concatenating
    # variable-length eval batches; decoding a negative id raises OverflowError.
    pred_ids  = np.where(pred_ids  < 0, tokenizer.pad_token_id, pred_ids)
    label_ids = np.where(label_ids < 0, tokenizer.pad_token_id, label_ids)
    preds = [dec_txt(t) for t in tokenizer.batch_decode(pred_ids,  skip_special_tokens=False)]
    refs  = [dec_txt(t) for t in tokenizer.batch_decode(label_ids, skip_special_tokens=False)]
    if preds:
        first = tokenizer.convert_ids_to_tokens([int(pred_ids[0][0])])[0]
        print(f"\n--- eval sample ---  first generated token: {first!r} "
              f"(expected {TASK_START_TOKEN!r})")
        print("PRED:", preds[0][:300])
        print("REF :", refs[0][:300])
        print("pairs pred/ref:", len(parse_pairs(preds[0])), "/", len(parse_pairs(refs[0])))
        print("--- end ---\n")
    return score_predictions(preds, refs)

print("metrics: precision, recall, f1, exact_match, edit_similarity, raw_similarity")
print("raw_similarity is the continuous one -- watch it move off 0 first.")
print("=" * 70)


In [ ]:
# ============================================================
# CELL 7 — TRAINING
# ============================================================
from transformers import (EarlyStoppingCallback, Seq2SeqTrainer,
                          Seq2SeqTrainingArguments, TrainerCallback)

print("=" * 70); print("TRAINING"); print("=" * 70)

class AssertGenerationConfig(TrainerCallback):
    """Trainer._align_special_tokens() rewrites generation_config at train start
    (the 'Updated tokens: {bos_token_id: 0}' log line). Re-assert the task token
    before training and before every evaluation so it cannot be clobbered."""
    def _fix(self, model):
        if model is None:
            return
        model.generation_config.decoder_start_token_id = TASK_START_ID
        model.generation_config.bos_token_id = TASK_START_ID
        model.generation_config.no_repeat_ngram_size = 0
        model.generation_config.repetition_penalty = 1.0
    def on_train_begin(self, args, state, control, model=None, **kw):
        self._fix(model)
    def on_evaluate(self, args, state, control, model=None, **kw):
        self._fix(model)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=60,
    per_device_train_batch_size=1,        # [1280,960] is memory-heavy on a T4
    gradient_accumulation_steps=8,        # effective batch 8 (official CORD uses 8)
    learning_rate=3e-5,                   # official Donut CORD value
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    eval_strategy="epoch",
    per_device_eval_batch_size=1,
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,     # never below the target distribution
    generation_num_beams=1,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="loss",         # task metrics can sit at 0 early
    greater_is_better=False,
    logging_strategy="steps", logging_steps=10,
    report_to="none",
    dataloader_num_workers=2,
    remove_unused_columns=False,
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=validation_dataset,
    processing_class=processor, compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=15),
               AssertGenerationConfig()],
)

steps = -(-len(train_dataset) // 8)
print(f"variant={MODEL_VARIANT}  epochs={training_args.num_train_epochs}  "
      f"steps/epoch~{steps}  total~{steps*training_args.num_train_epochs}")
print(f"image_size={IMAGE_SIZE}  gen_max_length={MAX_LENGTH}")
print(f"decoder_start={model.generation_config.decoder_start_token_id} "
      f"(expect {TASK_START_ID})  no_repeat_ngram={model.generation_config.no_repeat_ngram_size} "
      f"rep_penalty={model.generation_config.repetition_penalty}")
print("=" * 70)

train_result = trainer.train()

print("\n" + "=" * 70); print("SAVING"); print("=" * 70)
trainer.save_model(FINAL_MODEL_PATH)
processor.save_pretrained(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)
for k, v in train_result.metrics.items():
    print(f"  {k}: {v}")
print(f"  GPU allocated {torch.cuda.memory_allocated()/1024**3:.2f} GB / "
      f"reserved {torch.cuda.memory_reserved()/1024**3:.2f} GB")
print("saved to", FINAL_MODEL_PATH)
print("=" * 70)


In [ ]:
# ============================================================
# CELL 8 — FINAL VALIDATION + REPORT
# ============================================================
print("=" * 70); print("FINAL VALIDATION"); print("=" * 70)

model.eval()
model.generation_config.decoder_start_token_id = TASK_START_ID
model.generation_config.bos_token_id = TASK_START_ID

preds, refs = [], []
for i in range(len(validation_dataset)):
    s = validation_dataset[i]
    with torch.no_grad():
        out = model.generate(
            s["pixel_values"].unsqueeze(0).cuda(),
            max_length=MAX_LENGTH,
            decoder_start_token_id=TASK_START_ID,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            bad_words_ids=[[tokenizer.unk_token_id]],
            num_beams=1, do_sample=False, use_cache=True,
        )
    preds.append(dec_txt(tokenizer.decode(out[0], skip_special_tokens=False)))
    lab = s["labels"].clone(); lab[lab == -100] = tokenizer.pad_token_id
    refs.append(dec_txt(tokenizer.decode(lab, skip_special_tokens=False)))

metrics = score_predictions(preds, refs)
print(f"\ngenerated {len(preds)} predictions\n")
print("=" * 50)
for k, v in metrics.items():
    print(f"  {k:16s}: {v:.4f}")
print("=" * 50)

print("\nqualitative samples:")
for i in range(min(3, len(preds))):
    pp, rp = parse_pairs(preds[i]), parse_pairs(refs[i])
    print(f"\n--- sample {i} --- pred {len(pp)} pairs / ref {len(rp)} pairs")
    for q, a in pp[:4]:
        print(f"   PRED  {q!r} -> {a!r}")
    for q, a in rp[:4]:
        print(f"   REF   {q!r} -> {a!r}")

report = {"model_variant": MODEL_VARIANT, "dataset": DATASET_KIND,
          "image_size": IMAGE_SIZE, "max_length": MAX_LENGTH,
          "vocab_size": len(tokenizer), "metrics": metrics,
          "samples": [{"index": i, "predicted": parse_pairs(preds[i]),
                       "reference": parse_pairs(refs[i]),
                       "raw_prediction": preds[i][:2000]}
                      for i in range(len(preds))]}
os.makedirs(FINAL_MODEL_PATH, exist_ok=True)
path = os.path.join(FINAL_MODEL_PATH, "validation_report.json")
with open(path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print("\nsaved", path)
print("Bring this file back with the notebook + logs -- see Guide/rules.md.")
print("=" * 70)
